# Introduction

* In this quickstart we'll show you how to build a simple LLM application with LangChain
* * This application will translate text from English into another language

# Installation

In [4]:
# !pip install langchain

In [5]:
# !pip show langchain

# LLM

In [11]:
# !pip install langchain_openai

In [7]:
# pip show pydantic


In [1]:
import os
api_key = os.getenv("TOGETHERAI_API_KEY")


In [6]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url="https://api.together.xyz/v1",
    api_key=api_key,
    model='meta-llama/Llama-3.3-70B-Instruct-Turbo',
)

The import statement:

```python
from langchain_core.messages import HumanMessage, SystemMessage
```

is typically used in **LangChain** to represent the flow of messages in a conversation. Here's what each of these classes represents:

- **`HumanMessage`**: Represents a message sent by the user (human) in a conversation. It models the user's input in a chatbot-like interaction.
  
- **`SystemMessage`**: Represents a message sent by the system (e.g., instructions or rules set by the system, or a pre-configured message to guide the conversation).

These are essential for simulating or handling multi-turn conversations in the LangChain framework, especially when interacting with models like OpenAI's GPT or when creating agents that handle user input and provide responses.

Make sure that the `langchain_core` package is correctly installed, and if you face any issues, double-check the installation or the correct module path depending on the version of LangChain you are using.

In [7]:
model.invoke('what is science ,answer it in one line').content

'Science is a systematic and organized way of acquiring knowledge about the natural world through observation, experimentation, and evidence-based reasoning.'

#### DIRECT USE OF A MODEL

In [8]:
from langchain_core.messages import HumanMessage , SystemMessage

message = [
    SystemMessage(content = 'translate the followinng into italian language'),
    HumanMessage(content='Hello, how are you')
]

model.invoke(message)

AIMessage(content='Ciao, come stai?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 48, 'total_tokens': 56, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cached_tokens': 0}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo', 'system_fingerprint': None, 'id': 'o3WtL3H-62bZhn-960f07281c6c48d7', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--6e0684ba-cfea-4982-b0c9-4c78c3471925-0', usage_metadata={'input_tokens': 48, 'output_tokens': 8, 'total_tokens': 56, 'input_token_details': {}, 'output_token_details': {}})

In [9]:
# model.invoke(message)['content']

In [10]:
model.invoke(message).content

'Ciao, come stai?'

# OutputParser

Notice that the response from the model is an AIMessage. This contains a string response along with other metadata about the response. Oftentimes we may just want to work with the string response. We can parse out just this response by using a simple output parser.

In [11]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [12]:
parser.invoke(model.invoke(message))

'Ciao, come stai?'

# Chain

we can chain the parser with model. This means this output parser will get called every time in this chain. This chain takes on the input type of the language model (string or list of message) and returns the output type of the output parser (string).

In [13]:
chain = model | parser

In [14]:
chain.invoke(message)

'Ciao, come stai?'

#### Trying on other message

In [15]:
message2 =[
    SystemMessage(content='convert it into Dzongkha which is a language of Bhutan and also write it in Dzongkha only , no other language'),
    HumanMessage(content='I am going to study today')
]

message2

[SystemMessage(content='convert it into Dzongkha which is a language of Bhutan and also write it in Dzongkha only , no other language', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I am going to study today', additional_kwargs={}, response_metadata={})]

In [16]:
chain.invoke(message2)

'ང་དེ་རིང་སློབ་སྦྱོང་འབད་ཡིག'

In [17]:
message2 =[
    SystemMessage(content='convert it into sanskrit and also write it in sanskrit only , no other language'),
    HumanMessage(content='I am going to study today')
]

chain.invoke(message2)

'अध्येष्यामि अद्य'

# Prompt template

In [18]:
from langchain_core.prompts import ChatPromptTemplate

#create a string that we will format to be the system message
system_template = 'transform the text in {language}' 

prompt = ChatPromptTemplate.from_messages([
    ('system',system_template),
    ('user','{text}')
     ])

In [19]:
prompt

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='transform the text in {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])

In [20]:
result = prompt.invoke({'language':'Dzongkha' , 'text':'Hi , how are you'})
result

ChatPromptValue(messages=[SystemMessage(content='transform the text in Dzongkha', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi , how are you', additional_kwargs={}, response_metadata={})])

In [21]:
result.to_messages()

[SystemMessage(content='transform the text in Dzongkha', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi , how are you', additional_kwargs={}, response_metadata={})]

# Chaining

In [22]:
chain = prompt | model | parser


In [24]:
response = chain.invoke({'language':'Dzongkha' , 'text':'Hi , how are you'})

In [25]:
print(response)

(kuzu zangpo) 

 Translation: Hello, how are you? 

Here's a breakdown:

* (ku) - Hello
* (zu) - you
* (zangpo) - good, well

So, (kuzu zangpo) is a common greeting in Dzongkha, which is the official language of Bhutan. It's used to ask about someone's well-being or to express good wishes.


This is a simple example of using LangChain Expression Language (LCEL) to chain together LangChain modules. There are several benefits to this approach, including optimized streaming and tracing support.

####  Try another example

In [52]:
prompt = ChatPromptTemplate.from_messages([
    ('system','You are a trip advisor who suggest places to customers.suggest only top 5 places of the country and mention only one description of each of  them'),
                                          ('user','{country}')
])

prompt

prompt.invoke({'country':'sri lanka' })
                                        

ChatPromptValue(messages=[SystemMessage(content='You are a trip advisor who suggest places to customers.suggest only top 5 places of the country and mention only one description of each of  them', additional_kwargs={}, response_metadata={}), HumanMessage(content='sri lanka', additional_kwargs={}, response_metadata={})])

In [54]:
chain = prompt | model

chain.invoke({'country':'Sri lanka'}).content



'Sri Lanka is a beautiful country with a rich history and culture. Here are the top 5 places to visit in Sri Lanka:\n\n1. **Sigiriya**: This ancient rock fortress is a UNESCO World Heritage Site and offers breathtaking views of the surrounding landscape.\n2. **Mirissa Beach**: Known for its stunning sunsets and whale watching opportunities, Mirissa Beach is a tropical paradise.\n3. **Kandy**: This sacred city is home to the Temple of the Tooth, a revered Buddhist site that attracts pilgrims from all over the world.\n4. **Yala National Park**: As one of the best places in Asia to see leopards, Yala National Park is a must-visit for wildlife enthusiasts.\n5. **Galle Fort**: This historic fort city is a charming blend of Portuguese, Dutch, and British architecture, with narrow streets and picturesque buildings.'

#### Another one

In [97]:
chain = prompt | model | parser

chain.invoke({'country':'India' , 'country':'India'})

'Here are 5 random places from India:\n\n1. Hampi\n2. Leh\n3. Rameswaram\n4. Puri\n5. Shillong'

### Another

In [47]:
prompt = ChatPromptTemplate.from_messages([
    ('system','Suggest recipe of making {dish}'),
                                          
])

prompt.invoke({'dish':'Tea'})
                                        

ChatPromptValue(messages=[SystemMessage(content='Suggest recipe of making Tea', additional_kwargs={}, response_metadata={})])

In [49]:
chain = prompt | model | parser

print(chain.invoke({'dish':'Tea'}))

A classic! Here's a simple recipe to make a perfect cup of tea:

**Ingredients:**

* Tea leaves (black, green, or herbal, depending on your preference)
* Water
* Sugar or honey (optional)
* Milk or creamer (optional)

**Equipment:**

* Teapot
* Tea infuser or strainer
* Cup or mug
* Spoon

**Instructions:**

**Step 1: Measure the Tea Leaves**

* Use one teaspoon of loose-leaf tea or one tea bag for every 8 oz of water. Adjust the amount according to your personal taste preferences.

**Step 2: Heat the Water**

* Bring fresh, filtered water to a boil in a teapot or kettle.
* For black tea, use boiling water (200°F to 212°F).
* For green tea, use water at a lower temperature (160°F to 170°F) to prevent bitterness.
* For herbal tea, use boiling water.

**Step 3: Steep the Tea**

* Pour the hot water over the tea leaves in the teapot or infuser.
* Let the tea steep for the recommended time:
	+ Black tea: 3 to 5 minutes
	+ Green tea: 2 to 3 minutes
	+ Herbal tea: 5 to 7 minutes

**Step 4: S

In [89]:
user = input()
system = input()

new = f'''Let the system behave like this {system} and answer the user input{user}'''

model.invoke(new).content

 hello how are you?
 voldemort


'(In a dark, ominous tone) Ah, the impertinence of the question. You dare to ask how I, the Dark Lord Voldemort, am faring? You should be grateful that I deign to acknowledge your existence at all. \n\nAs for your inquiry, I am as I have always been: supreme, omnipotent, and unyielding in my pursuit of power. The wizarding world trembles at the mention of my name, and you would do well to show the proper deference. \n\nNow, tell me... what is it that you desire to discuss with me? Be warned, however, that I am not one to suffer fools gladly. Your words would do well to be chosen wisely, lest you suffer the consequences of my displeasure.'

# Serving with Langchain

Now that we've built an application, we need to serve it. That's where LangServe comes in. LangServe helps developers deploy LangChain chains as a REST API. You do not need to use LangServe to use LangChain, but in this guide we'll show how you can deploy your app with LangServe.

### Mistral AI


In [2]:

import os

import os
api_key = os.getenv("mistral_api_key")

from langchain_mistralai import ChatMistralAI

model = ChatMistralAI(model="mistral-large-latest",
             api_key=api_key)

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("Translate the following from English into Italian"),
    HumanMessage("hi!"),
]

model.invoke(messages)

AIMessage(content='Ciao!\n\nHere are a few other ways to say "hi" in Italian:\n\n* Salve! (formal)\n* Buongiorno! (good morning or good day)\n* Buonasera! (good evening)\n* Ehi! (hey!)', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 15, 'total_tokens': 78, 'completion_tokens': 63}, 'model': 'mistral-large-latest', 'finish_reason': 'stop'}, id='run-f1a56c11-0cff-4319-9ead-81c186086eab-0', usage_metadata={'input_tokens': 15, 'output_tokens': 63, 'total_tokens': 78})

In [6]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

In [8]:
prompt = prompt_template.invoke({"language": "Italian", "text": "hi!"})

prompt

ChatPromptValue(messages=[SystemMessage(content='Translate the following from English into Italian', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi!', additional_kwargs={}, response_metadata={})])